In [ ]:
pip uninstall -y torch torchvision torchaudio

In [ ]:
pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!pip install numpy==1.26.4 --force-reinstall

In [ ]:
from functools import wraps
from packaging import version
from collections import namedtuple

import torch
from torch import nn, einsum
import torch.nn.functional as F

from einops import rearrange

# constants
AttentionConfig = namedtuple('AttentionConfig', ['backends'])

# helpers
def exists(val):
    return val is not None

def default(val, d):
    return val if exists(val) else d

def once(fn):
    called = False
    @wraps(fn)
    def inner(x):
        nonlocal called
        if called:
            return
        called = True
        return fn(x)
    return inner

print_once = once(print)

# main class
class Attend(nn.Module):
    def __init__(
        self,
        dropout=0.,
        flash=False,
        scale=None
    ):
        super().__init__()
        self.dropout = dropout
        self.scale = scale
        self.attn_dropout = nn.Dropout(dropout)

        self.flash = flash

        # chỉ giữ check version cơ bản
        assert not (flash and version.parse(torch.__version__) < version.parse('2.0.0')), \
            'Flash attention requires PyTorch >= 2.0'

    def flash_attn(self, q, k, v):
        # scale nếu có
        if exists(self.scale):
            default_scale = q.shape[-1]
            q = q * (self.scale / default_scale)

        # đảm bảo contiguous
        q, k, v = map(lambda t: t.contiguous(), (q, k, v))

        # dùng SDPA 
        out = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.
        )

        return out

    def forward(self, q, k, v):
        """
        b - batch
        h - heads
        i, j - sequence length
        d - feature dimension
        """

        if self.flash:
            return self.flash_attn(q, k, v)

        # fallback attention 
        scale = default(self.scale, q.shape[-1] ** -0.5)

        # similarity
        sim = einsum("b h i d, b h j d -> b h i j", q, k) * scale

        # attention weights
        attn = sim.softmax(dim=-1)
        attn = self.attn_dropout(attn)

        # aggregate
        out = einsum("b h i j, b h j d -> b h i d", attn, v)

        return out

In [ ]:
!pip install ema-pytorch

In [ ]:
import math
import copy
from pathlib import Path
from random import random
from functools import partial
from collections import namedtuple
from multiprocessing import cpu_count

import torch
from torch import nn, einsum
import torch.nn.functional as F
from torch.nn import Module, ModuleList
from torch.amp import autocast
from torch.utils.data import Dataset, DataLoader

from torch.optim import Adam

from torchvision import transforms as T, utils

from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange

from scipy.optimize import linear_sum_assignment

from PIL import Image
from tqdm.auto import tqdm
from ema_pytorch import EMA

from accelerate import Accelerator


In [ ]:
def exists(x):
    return x is not None

def default(val, d):
    if exists(val):
        return val
    return d() if callable(d) else d

def cast_tuple(t, length = 1):
    if isinstance(t, tuple):
        return t
    return ((t,) * length)

def divisible_by(numer, denom):
    return (numer % denom) == 0

def identity(t, *args, **kwargs):
    return t

def cycle(dl):
    while True:
        for data in dl:
            yield data

def has_int_squareroot(num):
    return (math.sqrt(num) ** 2) == num

def num_to_groups(num, divisor):
    groups = num // divisor
    remainder = num % divisor
    arr = [divisor] * groups
    if remainder > 0:
        arr.append(remainder)
    return arr

def convert_image_to_fn(img_type, image):
    if image.mode != img_type:
        return image.convert(img_type)
    return image


In [ ]:
# normalization functions

def normalize_to_neg_one_to_one(img):
    return img * 2 - 1

def unnormalize_to_zero_to_one(t):
    return (t + 1) * 0.5


In [ ]:
def Upsample(dim, dim_out = None):
    return nn.Sequential(
        nn.Upsample(scale_factor = 2, mode = 'nearest'),
        nn.Conv2d(dim, default(dim_out, dim), 3, padding = 1)
    )

def Downsample(dim, dim_out = None):
    return nn.Sequential(
        Rearrange('b c (h p1) (w p2) -> b (c p1 p2) h w', p1 = 2, p2 = 2),
        nn.Conv2d(dim * 4, default(dim_out, dim), 1)
    )

class RMSNorm(Module):
    def __init__(self, dim):
        super().__init__()
        self.scale = dim ** 0.5
        self.g = nn.Parameter(torch.ones(1, dim, 1, 1))

    def forward(self, x):
        return F.normalize(x, dim = 1) * self.g * self.scale

# sinusoidal positional embeds

class SinusoidalPosEmb(Module):
    def __init__(self, dim, theta = 10000):
        super().__init__()
        self.dim = dim
        self.theta = theta

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(self.theta) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

class RandomOrLearnedSinusoidalPosEmb(Module):
    def __init__(self, dim, is_random = False):
        super().__init__()
        assert divisible_by(dim, 2)
        half_dim = dim // 2
        self.weights = nn.Parameter(torch.randn(half_dim), requires_grad = not is_random)

    def forward(self, x):
        x = rearrange(x, 'b -> b 1')
        freqs = x * rearrange(self.weights, 'd -> 1 d') * 2 * math.pi
        fouriered = torch.cat((freqs.sin(), freqs.cos()), dim = -1)
        fouriered = torch.cat((x, fouriered), dim = -1)
        return fouriered

# building block modules

class Block(Module):
    def __init__(self, dim, dim_out, dropout = 0.):
        super().__init__()
        self.proj = nn.Conv2d(dim, dim_out, 3, padding = 1)
        self.norm = RMSNorm(dim_out)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, scale_shift = None):
        x = self.proj(x)
        x = self.norm(x)

        if exists(scale_shift):
            scale, shift = scale_shift
            x = x * (scale + 1) + shift

        x = self.act(x)
        return self.dropout(x)

class ResnetBlock(Module):
    def __init__(self, dim, dim_out, *, time_emb_dim = None, dropout = 0.):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, dim_out * 2)
        ) if exists(time_emb_dim) else None

        self.block1 = Block(dim, dim_out, dropout = dropout)
        self.block2 = Block(dim_out, dim_out)
        self.res_conv = nn.Conv2d(dim, dim_out, 1) if dim != dim_out else nn.Identity()

    def forward(self, x, time_emb = None):

        scale_shift = None
        if exists(self.mlp) and exists(time_emb):
            time_emb = self.mlp(time_emb)
            time_emb = rearrange(time_emb, 'b c -> b c 1 1')
            scale_shift = time_emb.chunk(2, dim = 1)

        h = self.block1(x, scale_shift = scale_shift)

        h = self.block2(h)

        return h + self.res_conv(x)

class LinearAttention(Module):
    def __init__(
        self,
        dim,
        heads = 4,
        dim_head = 32,
        num_mem_kv = 4
    ):
        super().__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        hidden_dim = dim_head * heads

        self.norm = RMSNorm(dim)

        self.mem_kv = nn.Parameter(torch.randn(2, heads, dim_head, num_mem_kv))
        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias = False)

        self.to_out = nn.Sequential(
            nn.Conv2d(hidden_dim, dim, 1),
            RMSNorm(dim)
        )

    def forward(self, x):
        b, c, h, w = x.shape

        x = self.norm(x)

        qkv = self.to_qkv(x).chunk(3, dim = 1)
        q, k, v = map(lambda t: rearrange(t, 'b (h c) x y -> b h c (x y)', h = self.heads), qkv)

        mk, mv = map(lambda t: repeat(t, 'h c n -> b h c n', b = b), self.mem_kv)
        k, v = map(partial(torch.cat, dim = -1), ((mk, k), (mv, v)))

        q = q.softmax(dim = -2)
        k = k.softmax(dim = -1)

        q = q * self.scale

        context = torch.einsum('b h d n, b h e n -> b h d e', k, v)

        out = torch.einsum('b h d e, b h d n -> b h e n', context, q)
        out = rearrange(out, 'b h c (x y) -> b (h c) x y', h = self.heads, x = h, y = w)
        return self.to_out(out)

class Attention(Module):
    def __init__(
        self,
        dim,
        heads = 4,
        dim_head = 32,
        num_mem_kv = 4,
        flash = False
    ):
        super().__init__()
        self.heads = heads
        hidden_dim = dim_head * heads

        self.norm = RMSNorm(dim)
        self.attend = Attend(flash = flash)

        self.mem_kv = nn.Parameter(torch.randn(2, heads, num_mem_kv, dim_head))
        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias = False)
        self.to_out = nn.Conv2d(hidden_dim, dim, 1)

    def forward(self, x):
        b, c, h, w = x.shape

        x = self.norm(x)

        qkv = self.to_qkv(x).chunk(3, dim = 1)
        q, k, v = map(lambda t: rearrange(t, 'b (h c) x y -> b h (x y) c', h = self.heads), qkv)

        mk, mv = map(lambda t: repeat(t, 'h n d -> b h n d', b = b), self.mem_kv)
        k, v = map(partial(torch.cat, dim = -2), ((mk, k), (mv, v)))

        out = self.attend(q, k, v)

        out = rearrange(out, 'b h (x y) d -> b (h d) x y', x = h, y = w)
        return self.to_out(out)


In [ ]:
import torch
from torch import nn
from torch.nn import Module, ModuleList
import torch.nn.functional as F
from functools import partial

# -----------------------------
# Helpers 
# -----------------------------
def default(val, d):
    if val is not None:
        return val
    return d() if callable(d) else d

def cast_tuple(val, length):
    return val if isinstance(val, tuple) else (val,) * length


# -----------------------------
# UNet cho Multi-modal
# -----------------------------
class Unet(Module):
    def __init__(
        self,
        dim,
        init_dim=None,
        out_dim=None,
        dim_mults=(1, 2, 4, 8),
        channels=3,            # Số kênh ảnh 
        num_classes=2,         # Số kênh mask 
        self_condition=False,
        learned_variance=False,
        learned_sinusoidal_cond=False,
        random_fourier_features=False,
        learned_sinusoidal_dim=16,
        sinusoidal_pos_emb_theta=10000,
        dropout=0.,
        attn_dim_head=32,
        attn_heads=4,
        full_attn=None,
        flash_attn=False
    ):
        super().__init__()

        self.img_channels = channels
        self.mask_channels = num_classes
        self.channels = channels + num_classes
        self.combined_channels = self.channels
        self.self_condition = self_condition

        input_channels = self.combined_channels * (2 if self_condition else 1)

        init_dim = default(init_dim, dim)
        self.init_conv = nn.Conv2d(input_channels, init_dim, 7, padding=3)

        dims = [init_dim, *map(lambda m: dim * m, dim_mults)]
        in_out = list(zip(dims[:-1], dims[1:]))

        time_dim = dim * 4
        self.random_or_learned_sinusoidal_cond = learned_sinusoidal_cond or random_fourier_features

        if self.random_or_learned_sinusoidal_cond:
            sinu_pos_emb = RandomOrLearnedSinusoidalPosEmb(learned_sinusoidal_dim, random_fourier_features)
            fourier_dim = learned_sinusoidal_dim + 1
        else:
            sinu_pos_emb = SinusoidalPosEmb(dim, theta=sinusoidal_pos_emb_theta)
            fourier_dim = dim

        self.time_mlp = nn.Sequential(
            sinu_pos_emb,
            nn.Linear(fourier_dim, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim)
        )

        # Attention settings
        if not full_attn:
            full_attn = (*((False,) * (len(dim_mults) - 1)), True)

        num_stages = len(dim_mults)
        full_attn = cast_tuple(full_attn, num_stages)
        attn_heads = cast_tuple(attn_heads, num_stages)
        attn_dim_head = cast_tuple(attn_dim_head, num_stages)

        FullAttention = partial(Attention, flash=flash_attn)
        resnet_block = partial(ResnetBlock, time_emb_dim=time_dim, dropout=dropout)

        self.downs = ModuleList([])
        self.ups = ModuleList([])
        num_resolutions = len(in_out)

        for ind, ((dim_in, dim_out), layer_full_attn, layer_attn_heads, layer_attn_dim_head) in enumerate(zip(in_out, full_attn, attn_heads, attn_dim_head)):
            is_last = ind >= (num_resolutions - 1)
            attn_klass = FullAttention if layer_full_attn else LinearAttention

            self.downs.append(ModuleList([
                resnet_block(dim_in, dim_in),
                resnet_block(dim_in, dim_in),
                attn_klass(dim_in, dim_head=layer_attn_dim_head, heads=layer_attn_heads),
                Downsample(dim_in, dim_out) if not is_last else nn.Conv2d(dim_in, dim_out, 3, padding=1)
            ]))

        mid_dim = dims[-1]
        self.mid_block1 = resnet_block(mid_dim, mid_dim)
        self.mid_attn = FullAttention(mid_dim, heads=attn_heads[-1], dim_head=attn_dim_head[-1])
        self.mid_block2 = resnet_block(mid_dim, mid_dim)

        for ind, ((dim_in, dim_out), layer_full_attn, layer_attn_heads, layer_attn_dim_head) in enumerate(zip(*map(reversed, (in_out, full_attn, attn_heads, attn_dim_head)))):
            is_last = ind == (len(in_out) - 1)
            attn_klass = FullAttention if layer_full_attn else LinearAttention

            self.ups.append(ModuleList([
                resnet_block(dim_out + dim_in, dim_out),
                resnet_block(dim_out + dim_in, dim_out),
                attn_klass(dim_out, dim_head=layer_attn_dim_head, heads=layer_attn_heads),
                Upsample(dim_out, dim_in) if not is_last else nn.Conv2d(dim_out, dim_in, 3, padding=1)
            ]))

        default_out_dim = (self.img_channels + self.mask_channels) * (1 if not learned_variance else 2)
        self.out_dim = default(out_dim, default_out_dim)

        self.final_res_block = resnet_block(init_dim * 2, init_dim)
        self.final_conv = nn.Conv2d(init_dim, self.out_dim, 1)

    @property
    def downsample_factor(self):
        return 2 ** (len(self.downs) - 1)

    def forward(self, x, time, x_self_cond=None):
        # time must be tensor of shape (batch,)
        if self.self_condition:
            x_self_cond = default(x_self_cond, lambda: torch.zeros_like(x))
            x = torch.cat((x_self_cond, x), dim=1)

        x = self.init_conv(x)
        r = x.clone()

        t = self.time_mlp(time)

        h = []
        for block1, block2, attn, downsample in self.downs:
            x = block1(x, t)
            h.append(x)
            x = block2(x, t)
            x = attn(x) + x
            h.append(x)
            x = downsample(x)

        x = self.mid_block1(x, t)
        x = self.mid_attn(x) + x
        x = self.mid_block2(x, t)

        for block1, block2, attn, upsample in self.ups:
            x = torch.cat((x, h.pop()), dim=1)
            x = block1(x, t)
            x = torch.cat((x, h.pop()), dim=1)
            x = block2(x, t)
            x = attn(x) + x
            x = upsample(x)

        x = torch.cat((x, r), dim=1)
        x = self.final_res_block(x, t)
        x = self.final_conv(x)

        img_out = x[:, :self.img_channels, :, :]
        mask_out = x[:, self.img_channels:self.img_channels + self.mask_channels, :, :]
        return torch.cat([img_out, mask_out], dim=1)

In [ ]:


def extract(a, t, x_shape):
    b, *_ = t.shape
    out = a.gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))

def linear_beta_schedule(timesteps):
    """
    linear schedule, proposed in original ddpm paper
    """
    scale = 1000 / timesteps
    beta_start = scale * 0.0001
    beta_end = scale * 0.02
    return torch.linspace(beta_start, beta_end, timesteps, dtype = torch.float64)

def cosine_beta_schedule(timesteps, s = 0.008):
    """
    cosine schedule
    as proposed in https://openreview.net/forum?id=-NEXDKk8gZ
    """
    steps = timesteps + 1
    t = torch.linspace(0, timesteps, steps, dtype = torch.float64) / timesteps
    alphas_cumprod = torch.cos((t + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)

def sigmoid_beta_schedule(timesteps, start = -3, end = 3, tau = 1, clamp_min = 1e-5):
    """
    sigmoid schedule
    proposed in https://arxiv.org/abs/2212.11972 - Figure 8
    better for images > 64x64, when used during training
    """
    steps = timesteps + 1
    t = torch.linspace(0, timesteps, steps, dtype = torch.float64) / timesteps
    v_start = torch.tensor(start / tau).sigmoid()
    v_end = torch.tensor(end / tau).sigmoid()
    alphas_cumprod = (-((t * (end - start) + start) / tau).sigmoid() + v_end) / (v_end - v_start)
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)


In [ ]:
ModelPrediction =  namedtuple('ModelPrediction', ['pred_noise', 'pred_x_start','mask_logits'])

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from einops import rearrange, reduce
from tqdm.auto import tqdm
from functools import partial
from random import random
from dataclasses import dataclass
import math


class GaussianDiffusion(nn.Module):
    def __init__(
        self,
        model,
        *,
        image_size,
        timesteps=1000,
        sampling_timesteps=None,
        objective='pred_x0',
        beta_schedule='cosine',
        schedule_fn_kwargs=dict(),
        ddim_sampling_eta=0.,
        auto_normalize=True,
        offset_noise_strength=0.,
        min_snr_loss_weight=False,
        min_snr_gamma=5,
        immiscible=False,
        snr_balance_b=0.684      # hệ số cân bằng 
    ):
        super().__init__()
        assert not (type(self) == GaussianDiffusion and model.channels != model.out_dim)
        assert not hasattr(model, 'random_or_learned_sinusoidal_cond') or not model.random_or_learned_sinusoidal_cond

        self.model = model
        self.img_channels = model.img_channels      # = 3
        self.mask_channels = model.mask_channels    # = 2
        self.channels = self.img_channels + self.mask_channels
        self.self_condition = model.self_condition
        self.snr_balance_b = snr_balance_b

        if isinstance(image_size, int):
            image_size = (image_size, image_size)
        assert isinstance(image_size, (tuple, list)) and len(image_size) == 2
        self.image_size = image_size

        self.objective = objective
        assert objective in {'pred_noise', 'pred_x0', 'pred_v'}

        # Chọn schedule
        if beta_schedule == 'linear':
            beta_schedule_fn = linear_beta_schedule
        elif beta_schedule == 'cosine':
            beta_schedule_fn = cosine_beta_schedule
        elif beta_schedule == 'sigmoid':
            beta_schedule_fn = sigmoid_beta_schedule
        else:
            raise ValueError(f'unknown beta schedule {beta_schedule}')

        betas = beta_schedule_fn(timesteps, **schedule_fn_kwargs)
        alphas = 1. - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.)

        timesteps, = betas.shape
        self.num_timesteps = int(timesteps)

        # sampling parameters
        self.sampling_timesteps = default(sampling_timesteps, timesteps)
        self.is_ddim_sampling = self.sampling_timesteps < timesteps
        self.ddim_sampling_eta = ddim_sampling_eta

        register_buffer = lambda name, val: self.register_buffer(name, val.to(torch.float32))

        register_buffer('betas', betas)
        register_buffer('alphas_cumprod', alphas_cumprod)
        register_buffer('alphas_cumprod_prev', alphas_cumprod_prev)

        register_buffer('sqrt_alphas_cumprod', torch.sqrt(alphas_cumprod))
        register_buffer('sqrt_one_minus_alphas_cumprod', torch.sqrt(1. - alphas_cumprod))
        register_buffer('log_one_minus_alphas_cumprod', torch.log(1. - alphas_cumprod))
        register_buffer('sqrt_recip_alphas_cumprod', torch.sqrt(1. / alphas_cumprod))
        register_buffer('sqrt_recipm1_alphas_cumprod', torch.sqrt(1. / alphas_cumprod - 1))

        # posterior q(x_{t-1} | x_t, x_0)
        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)
        register_buffer('posterior_variance', posterior_variance)
        register_buffer('posterior_log_variance_clipped', torch.log(posterior_variance.clamp(min=1e-20)))
        register_buffer('posterior_mean_coef1', betas * torch.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod))
        register_buffer('posterior_mean_coef2', (1. - alphas_cumprod_prev) * torch.sqrt(alphas) / (1. - alphas_cumprod))

        # offset noise & immiscible
        self.offset_noise_strength = offset_noise_strength
        self.immiscible = immiscible

        # loss weight 
        snr = alphas_cumprod / (1 - alphas_cumprod)
        maybe_clipped_snr = snr.clone()
        if min_snr_loss_weight:
            maybe_clipped_snr.clamp_(max=min_snr_gamma)

        if objective == 'pred_noise':
            register_buffer('loss_weight', maybe_clipped_snr / snr)
        elif objective == 'pred_x0':
            register_buffer('loss_weight', maybe_clipped_snr)
        elif objective == 'pred_v':
            register_buffer('loss_weight', maybe_clipped_snr / (snr + 1))

        # data normalization
        self.normalize = normalize_to_neg_one_to_one if auto_normalize else identity
        self.unnormalize = unnormalize_to_zero_to_one if auto_normalize else identity

    @property
    def device(self):
        return self.betas.device

    # ----- Các hàm predict -----
    def predict_start_from_noise(self, x_t, t, noise):
        return (extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t -
                extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise)

    def predict_noise_from_start(self, x_t, t, x0):
        return ((extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t - x0) /
                extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape))

    def predict_v(self, x_start, t, noise):
        return (extract(self.sqrt_alphas_cumprod, t, x_start.shape) * noise -
                extract(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape) * x_start)

    def predict_start_from_v(self, x_t, t, v):
        return (extract(self.sqrt_alphas_cumprod, t, x_t.shape) * x_t -
                extract(self.sqrt_one_minus_alphas_cumprod, t, x_t.shape) * v)

    def q_posterior(self, x_start, x_t, t):
        posterior_mean = (extract(self.posterior_mean_coef1, t, x_t.shape) * x_start +
                          extract(self.posterior_mean_coef2, t, x_t.shape) * x_t)
        posterior_variance = extract(self.posterior_variance, t, x_t.shape)
        posterior_log_variance_clipped = extract(self.posterior_log_variance_clipped, t, x_t.shape)
        return posterior_mean, posterior_variance, posterior_log_variance_clipped

    # ----- Model predictions cho multi‑modal -----
    def model_predictions(self, x, t, x_self_cond=None, clip_x_start=False, rederive_pred_noise=False):
        model_output = self.model(x, t, x_self_cond)   # [B, img_channels+mask_channels, H, W]

        # Tách các thành phần
        img_out = model_output[:, :self.img_channels, ...]
        mask_logits = model_output[:, self.img_channels:, ...]

        maybe_clip = partial(torch.clamp, min=-1., max=1.) if clip_x_start else identity

        if self.objective == 'pred_noise':
            # Dự đoán noise → tính x_start
            pred_noise = model_output
            x_start = self.predict_start_from_noise(x, t, pred_noise)

            # Tách img và mask từ x_start
            x_start_img = maybe_clip(x_start[:, :self.img_channels, ...])
            # Với mask, ta có logits từ x_start (vì x_start là x0 dự đoán)
            mask_logits_derived = x_start[:, self.img_channels:, ...]
            mask_probs = mask_logits_derived.softmax(dim=1)
            x_start_mask = (mask_probs * 2 - 1) * self.snr_balance_b

        elif self.objective == 'pred_x0':
            # Model trực tiếp dự đoán x_start
            x_start = model_output
            x_start_img = maybe_clip(x_start[:, :self.img_channels, ...])
            mask_probs = mask_logits.softmax(dim=1)
            x_start_mask = (mask_probs * 2 - 1) * self.snr_balance_b

        elif self.objective == 'pred_v':
            # Dự đoán v → tính x_start
            v = model_output
            x_start = self.predict_start_from_v(x, t, v)
            x_start_img = maybe_clip(x_start[:, :self.img_channels, ...])
            mask_logits_derived = x_start[:, self.img_channels:, ...]
            mask_probs = mask_logits_derived.softmax(dim=1)
            x_start_mask = (mask_probs * 2 - 1) * self.snr_balance_b

        else:
            raise ValueError(f'unknown objective {self.objective}')

        # Ghép lại x_start đã xử lý (img trong [-1,1], mask trong [-b, b])
        x_start_final = torch.cat((x_start_img, x_start_mask), dim=1)

        # Nếu cần, tính lại pred_noise từ x_start_final (dùng cho DDIM)
        if rederive_pred_noise:
            pred_noise = self.predict_noise_from_start(x, t, x_start_final)
        else:
            pred_noise = None

        return ModelPrediction(pred_noise, x_start_final, mask_logits)

    def p_mean_variance(self, x, t, x_self_cond=None, clip_denoised=True):
        preds = self.model_predictions(x, t, x_self_cond)
        x_start = preds.pred_x_start

        if clip_denoised:
            # Chỉ clip phần ảnh, phần mask đã được rescale bằng b
            x_start[:, :self.img_channels, ...].clamp_(-1., 1.)

        model_mean, posterior_variance, posterior_log_variance = self.q_posterior(x_start=x_start, x_t=x, t=t)
        return model_mean, posterior_variance, posterior_log_variance, x_start

    # ----- Sampling -----
    @torch.inference_mode()
    def p_sample(self, x, t: int, x_self_cond=None):
        b, *_, device = *x.shape, self.device
        batched_times = torch.full((b,), t, device=device, dtype=torch.long)
        model_mean, _, model_log_variance, x_start = self.p_mean_variance(
            x=x, t=batched_times, x_self_cond=x_self_cond, clip_denoised=True)
        noise = torch.randn_like(x) if t > 0 else 0.
        pred_img = model_mean + (0.5 * model_log_variance).exp() * noise
        return pred_img, x_start

    @torch.inference_mode()
    def p_sample_loop(self, shape, return_all_timesteps=False):
        batch, device = shape[0], self.device
        img = torch.randn(shape, device=device)
        imgs = [img]
        x_start = None
        for t in tqdm(reversed(range(0, self.num_timesteps)), desc='sampling loop', total=self.num_timesteps):
            self_cond = x_start if self.self_condition else None
            img, x_start = self.p_sample(img, t, self_cond)
            imgs.append(img)
        ret = img if not return_all_timesteps else torch.stack(imgs, dim=1)
        return ret

    @torch.inference_mode()
    def ddim_sample(self, shape, return_all_timesteps=False):
        batch, device = shape[0], self.device
        total_timesteps = self.num_timesteps
        sampling_timesteps = self.sampling_timesteps
        eta = self.ddim_sampling_eta

        times = torch.linspace(-1, total_timesteps - 1, steps=sampling_timesteps + 1)
        times = list(reversed(times.int().tolist()))
        time_pairs = list(zip(times[:-1], times[1:]))

        img = torch.randn(shape, device=device)
        imgs = [img]
        x_start = None
        final_mask_logits = None

        for time, time_next in tqdm(time_pairs, desc='ddim sampling loop'):
            time_cond = torch.full((batch,), time, device=device, dtype=torch.long)
            self_cond = x_start if self.self_condition else None
            preds = self.model_predictions(
                img, time_cond, self_cond,
                clip_x_start=True,
                rederive_pred_noise=True
            )
            x_start = preds.pred_x_start
            pred_noise = preds.pred_noise
            final_mask_logits = preds.mask_logits

            if time_next < 0:
                img = x_start
                imgs.append(img)
                break

            alpha = self.alphas_cumprod[time]
            alpha_next = self.alphas_cumprod[time_next]

            sigma = eta * ((1 - alpha / alpha_next) * (1 - alpha_next) / (1 - alpha)).sqrt()
            c = (1 - alpha_next - sigma ** 2).sqrt()

            noise = torch.randn_like(img)
            img = x_start * alpha_next.sqrt() + c * pred_noise + sigma * noise
            imgs.append(img)

        ret = img if not return_all_timesteps else torch.stack(imgs, dim=1)
        return ret,final_mask_logits

    @torch.inference_mode()
    def sample(self, batch_size=16, return_all_timesteps=False):
        h, w = self.image_size
        shape = (batch_size, self.channels, h, w)
        sample_fn = self.p_sample_loop if not self.is_ddim_sampling else self.ddim_sample
        result = sample_fn(shape, return_all_timesteps=return_all_timesteps)

        if isinstance(result, tuple):
            out, mask_logits = result
        else:
            out = result
            mask_logits = None

        # out là tensor [B, C, H, W] hoặc [B, T, C, H, W]
        if return_all_timesteps:
            # Nếu trả về tất cả các bước, unnormalize cho từng bước
            # Chỉ áp dụng cho kênh ảnh (img_channels đầu)
            out_img = out[..., :self.img_channels, :, :]   # [B, T, img_c, H, W]  
            out_img = self.unnormalize(out_img)
            if mask_logits is not None:
                out_mask = mask_logits  # logits
            else:
                out_mask = out[..., self.img_channels:, :, :]  # fallback
            return out_img, out_mask
        else:
            out_img = out[:, :self.img_channels, ...]
            out_img = self.unnormalize(out_img)
            if mask_logits is not None:
               return out_img, mask_logits
            else:
               out_mask = out[:, self.img_channels:, ...]
            return out_img, out_mask

    # ----- Các hàm phụ trợ khác-----
    def noise_assignment(self, x_start, noise):
        # Chỉ dùng khi immiscible=True
        try:
            from scipy.optimize import linear_sum_assignment
        except ImportError:
            raise ImportError("Please install scipy for noise_assignment")
        x_start_flat, noise_flat = tuple(rearrange(t, 'b ... -> b (...)') for t in (x_start, noise))
        dist = torch.cdist(x_start_flat, noise_flat)
        _, assign = linear_sum_assignment(dist.cpu())
        return torch.from_numpy(assign).to(dist.device)

    @torch.cuda.amp.autocast(enabled=False)
    def q_sample(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        if self.immiscible:
            assign = self.noise_assignment(x_start, noise)
            noise = noise[assign]
        return (extract(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start +
                extract(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape) * noise)

    # ----- Training loss  -----
    def p_losses(self, x_start, mask_one_hot, t, noise=None):
        """
        x_start:      [B, img_channels, H, W]  
        mask_one_hot: [B, mask_channels, H, W] 
        """
        # Chuyển mask sang không gian [-b, b]
        mask_continuous = (mask_one_hot * 2 - 1) * self.snr_balance_b
        combined_start = torch.cat((x_start, mask_continuous), dim=1)

        if noise is None:
            noise = torch.randn_like(combined_start)

        # offset noise (nếu có)
        if self.offset_noise_strength > 0.:
            offset_noise = torch.randn(combined_start.shape[:2], device=self.device)
            noise += self.offset_noise_strength * rearrange(offset_noise, 'b c -> b c 1 1')

        x_t = self.q_sample(x_start=combined_start, t=t, noise=noise)

        # self‑conditioning (50% chance)
        x_self_cond = None
        if self.self_condition and random() < 0.5:
            with torch.no_grad():
                preds = self.model_predictions(x_t, t, clip_x_start=True)
                x_self_cond = preds.pred_x_start.detach()

        model_out = self.model(x_t, t, x_self_cond)

        # Tách dự đoán
        pred_img, pred_mask_logits = torch.split(model_out, [self.img_channels, self.mask_channels], dim=1)

        # Xác định target cho ảnh
        if self.objective == 'pred_noise':
            target_img = noise[:, :self.img_channels]
        elif self.objective == 'pred_x0':
            target_img = x_start
        elif self.objective == 'pred_v':
            v_target = self.predict_v(combined_start, t, noise)
            target_img = v_target[:, :self.img_channels]
        else:
            raise ValueError(f'unknown objective {self.objective}')

        # Loss ảnh (MSE)
        loss_img = F.mse_loss(pred_img, target_img, reduction='none')
        loss_img = reduce(loss_img, 'b ... -> b', 'mean')

        # Loss mask (Cross Entropy)
        mask_target_idx = mask_one_hot.argmax(dim=1).long()
        loss_mask = F.cross_entropy(pred_mask_logits, mask_target_idx, reduction='none')
        loss_mask = reduce(loss_mask, 'b ... -> b', 'mean')

        # Áp dụng SNR weighting
        loss_weight = extract(self.loss_weight, t, loss_img.shape)

        total_loss = (loss_img + loss_mask) 
        return total_loss.mean()

    def forward(self, img, mask, *args, **kwargs):
        b, c, h, w = img.shape
        device = img.device
        assert h == self.image_size[0] and w == self.image_size[1], f'image size must be {self.image_size}'
        t = torch.randint(0, self.num_timesteps, (b,), device=device).long()

        img = self.normalize(img)
        return self.p_losses(img, mask, t, *args, **kwargs)

In [ ]:
import os
import tensorflow as tf

class ImageMaskDataset:
    def __init__(self, root_dir, resolution=128, mask_channels=2):
        self.root_dir = root_dir
        self.resolution = resolution
        self.mask_channels = mask_channels

    def _get_paths(self, split):
        img_dir = os.path.join(self.root_dir, split, "images")
        mask_dir = os.path.join(self.root_dir, split, "masks")

        img_paths = sorted(tf.io.gfile.glob(os.path.join(img_dir, "*.JPG")))
        mask_paths = sorted(tf.io.gfile.glob(os.path.join(mask_dir, "*.PNG")))

        assert len(img_paths) > 0, f"Không tìm thấy file .JPG tại {img_dir}"
        assert len(img_paths) == len(mask_paths), f"Lệch số lượng: {len(img_paths)} ảnh vs {len(mask_paths)} masks"
        return img_paths, mask_paths

    def _parse(self, img_path, mask_path, augment_flip=False):
        # 1. XỬ LÝ ẢNH
        img = tf.io.read_file(img_path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [self.resolution, self.resolution])
        img = tf.cast(img, tf.float32) / 255.0

        # XỬ LÝ MASK
        mask = tf.io.read_file(mask_path)
        mask = tf.image.decode_png(mask, channels=1)
        mask = tf.image.resize(
            mask, 
            [self.resolution, self.resolution], 
            method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
        )
        mask = tf.cast(mask > 0, tf.int32) 
        mask = tf.squeeze(mask, axis=-1)  # [H, W]
        mask_one_hot = tf.one_hot(mask, depth=self.mask_channels)  # [H, W, C]
        mask_one_hot = tf.cast(mask_one_hot, tf.float32)

        # AUGMENTATION: Horizontal Flip 
        if augment_flip and tf.random.uniform(()) > 0.5:
            img = tf.image.flip_left_right(img)
            mask_one_hot = tf.image.flip_left_right(mask_one_hot)

        # ĐỔI THỨ TỰ KÊNH (PyTorch style: [C, H, W])
        img = tf.transpose(img, [2, 0, 1])
        mask_one_hot = tf.transpose(mask_one_hot, [2, 0, 1])

        return img, mask_one_hot

    def build(self, split, batch_size, shuffle=True, seed=42, repeat=True, augment_flip=True):
        img_paths, mask_paths = self._get_paths(split)
        ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))

        if shuffle:
            ds = ds.shuffle(len(img_paths), seed=seed)

        if repeat:
            ds = ds.repeat()

        # Truyền tham số 
        def parse_with_augment(img_path, mask_path):
            return self._parse(img_path, mask_path, augment_flip=augment_flip)

        ds = ds.map(parse_with_augment, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=True)
        ds = ds.prefetch(tf.data.AUTOTUNE)

        return ds

In [ ]:
dataset_builder = ImageMaskDataset(
    root_dir='/kaggle/input/datasets/minhlm2023/otu2d-dataset/dataset_split',
    resolution=128
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tạo UNet model
unet = Unet(
    dim=64,
    channels=3,       # image channels
    num_classes=2,    # mask classes
    self_condition=False
).to(device)
print("UNET channels:", unet.channels)

In [ ]:
diffusion_model = GaussianDiffusion(
    model=unet,
    image_size=128,
    timesteps=1000
).to(device)

In [ ]:

trainer = Trainer(
    diffusion_model=diffusion_model,
    dataset_builder=dataset_builder,   
    train_batch_size=16,              
    gradient_accumulate_every=1,
    train_lr=2e-5,                     
    train_num_steps=100000,             
    ema_update_every=20,
    ema_decay=0.995,
    adam_betas=(0.9, 0.99),
    save_and_sample_every=2000,
    num_samples=9,
    results_folder='/kaggle/working/results',
    amp=True,
    mixed_precision_type='fp16',
    calculate_fid=False,
    calculate_sfid=False,
    num_fid_samples=1000,
    image_size=128,                    
    snr_balance_b=0.684
)



In [ ]:
import math
import torch
from pathlib import Path
from torch.optim import AdamW
from torchvision import utils
from accelerate import Accelerator
from ema_pytorch import EMA
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt

from queue import Queue
from threading import Thread

# tăng tốc cudnn
torch.backends.cudnn.benchmark = True


def prefetch_generator(generator, buffer_size=4):
    queue = Queue(maxsize=buffer_size)

    def worker():
        for item in generator:
            queue.put(item)
        queue.put(None)

    Thread(target=worker, daemon=True).start()

    while True:
        item = queue.get()
        if item is None:
            break
        yield item


class Trainer:
    def __init__(
        self,
        diffusion_model,
        dataset_builder,
        *,
        train_batch_size=16,
        gradient_accumulate_every=1,
        train_lr=2e-5,
        train_num_steps=30000,
        ema_update_every=20,          
        ema_decay=0.995,
        adam_betas=(0.9, 0.99),
        save_and_sample_every=2000,   
        num_samples=25,
        results_folder='/kaggle/working/results',
        amp=True,
        mixed_precision_type='fp16',
        calculate_fid=True,
        calculate_sfid=True,
        num_fid_samples=1000,       
        inception_block_idx=2048,
        image_size=128,
        snr_balance_b=0.684
    ):
        super().__init__()

        # ---------------- Accelerator ----------------
        self.accelerator = Accelerator(
            mixed_precision=mixed_precision_type if amp else 'no'
        )

        self.device = self.accelerator.device

        # ---------------- Model ----------------
        self.model = diffusion_model

        try:
            self.model = torch.compile(self.model)
        except:
            pass

        self.image_size = image_size
        self.snr_balance_b = snr_balance_b

        if isinstance(self.image_size, int):
           self.model.image_size = (self.image_size, self.image_size)
        else:
           self.model.image_size = self.image_size
        self.model.snr_balance_b = self.snr_balance_b

        self.channels = diffusion_model.channels
        self.mask_channels = diffusion_model.mask_channels
        self.img_channels = diffusion_model.img_channels

        # ---------------- Training config ----------------
        self.num_samples = num_samples
        self.save_and_sample_every = save_and_sample_every
        self.batch_size = train_batch_size
        self.gradient_accumulate_every = gradient_accumulate_every
        self.train_num_steps = train_num_steps

        # ---------------- Dataset ----------------
        tf_ds = dataset_builder.build(split='train', batch_size=train_batch_size)

        # ZERO-COPY LOADER
        def tf_to_pytorch_loader():
            for imgs_np, masks_np in tf_ds.as_numpy_iterator():
                imgs = torch.from_numpy(imgs_np.copy()).float()
                masks = torch.from_numpy(masks_np.copy()).float()

                if imgs.ndim == 4 and imgs.shape[-1] <= 3: 
                    imgs = imgs.permute(0, 3, 1, 2)

                if masks.ndim == 3:
                    # (B, H, W) -> (B, 1, H, W)
                    masks = masks.unsqueeze(1)
                elif masks.ndim == 4 and masks.shape[-1] == self.mask_channels:
                    # (B, H, W, C) -> (B, C, H, W)
                    masks = masks.permute(0, 3, 1, 2)

                target_size = self.image_size
                if isinstance(target_size, int):
                    target_size = (target_size, target_size)

                if imgs.shape[-2:] != target_size:
                    imgs = torch.nn.functional.interpolate(
                        imgs, size=target_size, mode='bilinear', align_corners=False
                    )
                    masks = torch.nn.functional.interpolate(
                        masks, size=target_size, mode='nearest'
                    )

                yield imgs, masks

        #  PREFETCH
        self.dl = cycle(prefetch_generator(tf_to_pytorch_loader()))

        # ---------------- Optimizer ----------------
        self.opt = AdamW(
            self.model.parameters(), 
            lr=train_lr, 
            betas=adam_betas,
            weight_decay=1e-4  
        )

        # ---------------- Prepare ----------------
        self.model, self.opt = self.accelerator.prepare(self.model, self.opt)

        # ---------------- EMA ----------------
        if self.accelerator.is_main_process:
            self.ema = EMA(self.model, beta=ema_decay, update_every=ema_update_every)
            self.ema.to(self.device)

        # ---------------- Save ----------------
        self.results_folder = Path(results_folder)
        self.results_folder.mkdir(exist_ok=True)
        self.step = 0

        # ---------------- Metrics ----------------
        self.calculate_fid = calculate_fid and self.accelerator.is_main_process
        self.calculate_sfid = calculate_sfid and self.accelerator.is_main_process

        if self.calculate_fid:
            self.fid_scorer = FIDEvaluation(
                batch_size=self.batch_size,
                dl=self.dl,
                sampler=self.ema.ema_model,
                channels=self.channels,
                accelerator=self.accelerator,
                stats_dir=results_folder,
                device=self.device,
                num_fid_samples=num_fid_samples,
                inception_block_idx=inception_block_idx,
                mode="fid"
            )

        if self.calculate_sfid:
            self.sfid_scorer = FIDEvaluation(
                batch_size=self.batch_size,
                dl=self.dl,
                sampler=self.ema.ema_model,
                channels=self.mask_channels,
                accelerator=self.accelerator,
                stats_dir=results_folder,
                device=self.device,
                num_fid_samples=num_fid_samples,
                inception_block_idx=inception_block_idx,
                mode="sfid"
            )

    # ---------------- Save ----------------
    def save(self, milestone):
        if not self.accelerator.is_local_main_process:
            return

        data = {
            'step': self.step,
            'model': self.accelerator.get_state_dict(self.model),
            'opt': self.opt.state_dict(),
            'ema': self.ema.state_dict(),
            'scaler': self.accelerator.scaler.state_dict()
            if hasattr(self.accelerator, 'scaler') and self.accelerator.scaler
            else None,
        }

        torch.save(data, str(self.results_folder / f'model-{milestone}.pt'))

    # ---------------- Train ----------------
    def train(self):

        with tqdm(
            initial=self.step,
            total=self.train_num_steps,
            disable=not self.accelerator.is_main_process
        ) as pbar:

            while self.step < self.train_num_steps:

                self.model.train()
                total_loss = 0.0

                with self.accelerator.accumulate(self.model):

                    img, mask = next(self.dl)

                    # non_blocking GPU transfer
                    img = img.to(self.device, non_blocking=True)
                    mask = mask.to(self.device, non_blocking=True)

                    with self.accelerator.autocast():
                        loss = self.model(img, mask)
                        loss = loss / self.gradient_accumulate_every

                    self.accelerator.backward(loss)
                    total_loss += loss.item() * self.gradient_accumulate_every

                # log
                pbar.set_description(
                    f'loss: {total_loss:.4f} | b: {self.snr_balance_b}'
                )

                # optimize
                self.accelerator.clip_grad_norm_(self.model.parameters(), 1.0)
                self.opt.step()
                self.opt.zero_grad()

                self.step += 1

                # ---------------- EMA + Eval ----------------
                if self.accelerator.is_main_process:
                    self.ema.update()

                    if self.step != 0 and (self.step % self.save_and_sample_every == 0):
                        self.ema.ema_model.eval()

                        with torch.inference_mode():
                            milestone = self.step // self.save_and_sample_every
                            sampled_result = self.ema.ema_model.sample(batch_size=self.num_samples)
                            
                            if isinstance(sampled_result, tuple):
                                imgs, masks = sampled_result # masks lúc này là Logits từ model
                            else:
                                imgs = sampled_result[:, :self.img_channels]
                                masks = sampled_result[:, self.img_channels:]

                        #  Lưu ảnh 
                        utils.save_image(
                            imgs,
                            str(self.results_folder / f'sample-img-{milestone}.png'),
                            nrow=int(math.sqrt(self.num_samples))
                        )

                        # Tạo Heatmap 
                        
                        mask_probs = (masks / self.snr_balance_b + 1) / 2
                        mask_probs = mask_probs.clamp(0, 1).cpu().numpy()
                        batch_size_sampled = mask_probs.shape[0]
                        num_classes = self.mask_channels
                        
                        idx = 0
                        fig, axes = plt.subplots(1, num_classes, figsize=(num_classes * 4, 4))
                        if num_classes == 1: axes = [axes]
                        
                        for c in range(num_classes):
                            # Vẽ từng class với colormap 'jet' (xanh dương -> đỏ)
                            im = axes[c].imshow(mask_probs[idx, c], cmap='jet', vmin=0, vmax=1)
                            axes[c].set_title(f'Class {c} Probability')
                            axes[c].axis('off')
                            fig.colorbar(im, ax=axes[c], fraction=0.046, pad=0.04)
                        
                        plt.tight_layout()
                        plt.savefig(str(self.results_folder / f'sample-heatmap-{milestone}.png'))
                        plt.close()

                        # Lưu thêm mask tổng hợp (argmax) để đối chiếu 
                        mask_final = masks.argmax(dim=1).float() / (self.mask_channels - 1)
                        utils.save_image(
                            mask_final.unsqueeze(1),
                            str(self.results_folder / f'sample-mask-final-{milestone}.png'),
                            nrow=int(math.sqrt(self.num_samples))
                        )

                        # -------- Metrics --------
                        fid_val = None
                        sfid_val = None

                        if self.calculate_fid and milestone % 5 == 0:
                            fid_val = self.fid_scorer.fid_score()

                        if self.calculate_sfid and milestone % 5 == 0:
                            sfid_val = self.sfid_scorer.fid_score()

                        self.accelerator.print(
                            f"Step {self.step}: "
                            f"FID {fid_val if fid_val else 'skip'} | "
                            f"sFID {sfid_val if sfid_val else 'skip'}"
                        )

                        self.save(milestone)

                pbar.update(1)

        self.accelerator.print('training complete') 

In [ ]:
trainer.train()

In [ ]:
dataset_builder = ImageMaskDataset(
    root_dir='/kaggle/input/datasets/minhlm2023/otu2d-dataset/dataset_split',
    resolution=256
)

In [ ]:
import torch
from pathlib import Path


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

unet_2 = Unet(
    dim=64,
    dim_mults=(1,2,4,8),
    channels=3,
    num_classes=2,
    self_condition=False,
    flash_attn=False
).to(device)

# Tạo diffusion model với image_size=256
diffusion_model_256 = GaussianDiffusion(
    model=unet_256,
    image_size=256,
    timesteps=1000,  
    objective='pred_x0',
    beta_schedule='cosine',
    auto_normalize=True,
    snr_balance_b=0.684
).to(device)

# Load checkpoint 128x128
checkpoint_path_128 = "/kaggle/input/models/minhlm2023/dakter50-/pytorch/default/1/model-50.pt"
checkpoint = torch.load(checkpoint_path_128, map_location=device)
if 'ema' in checkpoint:
    model_state = checkpoint['ema']['ema_model'] if 'ema_model' in checkpoint['ema'] else checkpoint['ema']
elif 'model' in checkpoint:
    model_state = checkpoint['model']
else:
    model_state = checkpoint

new_state_dict = {}
for key, value in model_state.items():
    if key.startswith('module.'):
        key = key[7:]
    if key.startswith('online_model.'):
        key = key[13:]
    if key.startswith('ema_model.'):
        key = key[10:]
    new_state_dict[key] = value

missing, unexpected = diffusion_model_256.load_state_dict(new_state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

trainer = Trainer(
    diffusion_model=diffusion_model_256,
    dataset_builder=dataset_builder,   
    train_batch_size=4,               
    gradient_accumulate_every=1,
    train_lr=1e-5,                     
    train_num_steps=70000,             
    ema_update_every=20,
    ema_decay=0.995,
    adam_betas=(0.9, 0.99),
    save_and_sample_every=2000,
    num_samples=4,
    results_folder='/kaggle/working/results',
    amp=True,
    mixed_precision_type='fp16',
    calculate_fid=False,
    calculate_sfid=False,
    num_fid_samples=1000,
    image_size=256,                   
    snr_balance_b=0.684
)



In [ ]:
trainer.train()

In [ ]:
pip install torchmetrics torchvision pillow numpy tqdm

In [ ]:
import torch
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image


def load_diffusion_checkpoint(checkpoint_path, diffusion, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    if "ema" in checkpoint:
        ema_state = checkpoint["ema"]
        if "online_model" in ema_state:
            model_state = ema_state["online_model"]
        elif "ema_model" in ema_state:
            model_state = ema_state["ema_model"]
        else:
            model_state = ema_state
    elif "model" in checkpoint:
        model_state = checkpoint["model"]
    else:
        model_state = checkpoint

    if isinstance(model_state, dict) and "model" in model_state:
        model_state = model_state["model"]

    new_state_dict = {}

    for key, value in model_state.items():
        if key in ("initted", "step", "total_steps"):
            continue

        if key.startswith("module."):
            new_key = key[7:]
        elif key.startswith("online_model."):
            new_key = key[13:]
        elif key.startswith("ema_model."):
            new_key = key[10:]
        else:
            new_key = key

        new_state_dict[new_key] = value

    diffusion.load_state_dict(new_state_dict, strict=True)
    diffusion.to(device)
    diffusion.eval()

    if "step" in checkpoint:
        print(f"Loaded checkpoint at step {checkpoint['step']}")

    return diffusion


def save_image(tensor, path):
    """
    tensor: (C, H, W) trong khoảng [0, 1]
    """
    img = tensor.permute(1, 2, 0).cpu().numpy()
    img = (img * 255).astype(np.uint8)
    Image.fromarray(img).save(path)


def generate_and_save(
    diffusion,
    batch_size,
    num_samples,
    output_dir,
    device,
):
    output_dir = Path(output_dir)

    img_dir = output_dir / "images"
    hard_mask_dir = output_dir / "hard_masks"
    logits_dir = output_dir / "logits_npy"

    for d in [img_dir, hard_mask_dir, logits_dir]:
        d.mkdir(parents=True, exist_ok=True)

    generated = 0
    pbar = tqdm(total=num_samples, desc="Generating")

    while generated < num_samples:
        batch = min(batch_size, num_samples - generated)

        with torch.no_grad():
            imgs, mask_logits = diffusion.sample(
                batch_size=batch,
                return_all_timesteps=False,
            )

        for i in range(batch):
            idx = generated + i

            # Lưu ảnh sinh
            img = imgs[i]
            save_image(img, img_dir / f"{idx:06d}.jpg")

            # Lưu hard mask
            hard_mask = mask_logits[i].argmax(dim=0).cpu().numpy()

            num_classes = mask_logits.shape[1]

            if num_classes > 1:
                hard_mask = (
                    hard_mask * (255 // (num_classes - 1))
                ).astype(np.uint8)
            else:
                hard_mask = hard_mask.astype(np.uint8)

            Image.fromarray(hard_mask).save(
                hard_mask_dir / f"{idx:06d}.jpg"
            )

            # Lưu logits gốc (chưa qua Softmax)
            np.save(
                logits_dir / f"{idx:06d}.npy",
                mask_logits[i].cpu().numpy(),
            )

        generated += batch
        pbar.update(batch)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pbar.close()

    print(f"Generated {generated} samples into {output_dir}")

In [ ]:
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # ========== KHỞI TẠO MODEL==========
    unet = Unet(
        dim=64,
        dim_mults=(1,2,4,8),
        channels=3,
        num_classes=2,
        self_condition=False,
        flash_attn=False
    )
    
    diffusion = GaussianDiffusion(
        model=unet,
        image_size=256,
        timesteps=1000,
        sampling_timesteps=200,   
        objective='pred_x0',
        beta_schedule='cosine',
        auto_normalize=True,
        snr_balance_b=0.684
    )
    
    # Load checkpoint
    diffusion = load_diffusion_checkpoint(
        "/kaggle/input/models/minhlm2023/model35-2/pytorch/default/1/model-35.pt",
        diffusion,
        device
    )
    
    # Sinh ảnh
    generate_and_save(
        diffusion=diffusion,
        batch_size=8,
        num_samples=1177,
        output_dir="/kaggle/working/generated_dataset_full",
        device=device
    )

In [ ]:
import zipfile
from pathlib import Path
from tqdm.auto import tqdm

def zip_directory(source_dir, output_zip_path):
    """
    Nén toàn bộ thư mục source_dir thành file .zip.
    """
    source_dir = Path(source_dir)
    output_zip_path = Path(output_zip_path)
    
    # Thu thập tất cả file
    all_files = [f for f in source_dir.rglob('*') if f.is_file()]
    
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in tqdm(all_files, desc="Zipping"):
            arcname = file_path.relative_to(source_dir)  # giữ cấu trúc bên trong source_dir
            zipf.write(file_path, arcname)
    print(f" Zipped to {output_zip_path} (size: {output_zip_path.stat().st_size / 1024**2:.1f} MB)")

# ===== Sử dụng =====
if __name__ == "__main__":
    output_dir = "/kaggle/working/generated_dataset_full"   # Thư mục chứa ảnh đã sinh
    zip_path = "/kaggle/working/generated_sample.zip" # File zip đầu ra
    zip_directory(output_dir, zip_path)